# Netezza SQL → PySpark Inference Notebook

**Standalone inference notebook — run this after a Colab runtime restart.**

### Prerequisites
The training notebook (`netezza_to_pyspark_finetune.ipynb`) must have been run at least once  
and Cell 5 must have completed successfully. The LoRA adapter is loaded from **Google Drive**:

```
MyDrive/netezza-pyspark-model/lora_model/    ← saved by training Cell 5
```

### Steps
| Cell | What it does |
|---|---|
| 1 | Install inference dependencies |
| 2 | Mount Google Drive and load fine-tuned model |
| 3 | Run your Netezza SQL queries |


In [ ]:
# ============================================================
# CELL 1 — INSTALL INFERENCE DEPENDENCIES
# Minimal install — only what is needed to load the model
# and run generation. No training libraries required.
# After install, DO NOT restart the runtime — continue to Cell 2.
# ============================================================

# Unsloth (official Colab wheel — matches runtime CUDA version)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Core inference stack
!pip install -q \
    transformers \
    bitsandbytes \
    accelerate \
    peft

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Dependencies ready.")

In [ ]:
# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE & LOAD FINE-TUNED MODEL
# Mounts Drive, then loads the base model + LoRA adapter from
# the path saved by the training notebook's Cell 5.
# ============================================================

from unsloth import FastLanguageModel
from google.colab import drive
import torch
import os

# -----------------------------------------------------------
# 2a. Mount Google Drive
# The training notebook saved the LoRA adapter to:
#   MyDrive/netezza-pyspark-model/lora_model/
# This mount makes that path available at /content/drive/MyDrive/
# -----------------------------------------------------------
print("Mounting Google Drive ...")
drive.mount("/content/drive")

LORA_MODEL_PATH = "/content/drive/MyDrive/netezza-pyspark-model/lora_model"
MAX_SEQ_LENGTH  = 2048

# Verify the adapter exists before attempting to load
if not os.path.isdir(LORA_MODEL_PATH):
    raise FileNotFoundError(
        f"\nLoRA adapter not found at: {LORA_MODEL_PATH}\n"
        "Make sure you have run the training notebook (Cell 5) at least once.\n"
        f"Contents of netezza-pyspark-model/: "
        f"{os.listdir(os.path.dirname(LORA_MODEL_PATH)) if os.path.isdir(os.path.dirname(LORA_MODEL_PATH)) else 'directory not found'}"
    )

# -----------------------------------------------------------
# 2b. Load base model + LoRA adapter
# FastLanguageModel.from_pretrained() detects the adapter config
# automatically when pointed at the lora_model/ directory.
# -----------------------------------------------------------
print(f"\nLoading model from Drive: {LORA_MODEL_PATH} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = LORA_MODEL_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,       # Auto: float16 on T4
    load_in_4bit   = True,
)

# Switch to fast inference mode (2x speed, disables grad tracking)
FastLanguageModel.for_inference(model)
print("Model loaded and ready for inference.")

# -----------------------------------------------------------
# 2c. Prompt template — must match the one used during training
# -----------------------------------------------------------
INFERENCE_TEMPLATE = """\
Below is an instruction that describes a task, paired with an input that provides further context. \
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

def translate(netezza_sql: str, max_new_tokens: int = 512, temperature: float = 0.1) -> str:
    """
    Translate a Netezza SQL query to PySpark DataFrame code.

    Args:
        netezza_sql    : The raw Netezza SQL string to translate.
        max_new_tokens : Maximum tokens to generate (increase for long queries).
        temperature    : Lower = more deterministic. 0.1 is recommended for code.

    Returns:
        Generated PySpark code as a string.
    """
    prompt = INFERENCE_TEMPLATE.format(
        instruction="Translate the following legacy Netezza SQL query into optimised PySpark DataFrame code.",
        input=netezza_sql.strip(),
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = temperature,
            do_sample      = True,
            use_cache      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    # Strip the prompt tokens — return only the generated response
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


print("translate() helper is ready. Proceed to Cell 3.")


In [ ]:
# ============================================================
# CELL 3 — RUN YOUR NETEZZA SQL QUERIES
# Add or edit entries in MY_QUERIES below.
# Each entry is a dict with:
#   label : short description shown in the output header
#   sql   : the raw Netezza SQL to translate
# ============================================================

MY_QUERIES = [
    {
        "label": "Aggregation with CASE WHEN",
        "sql": """\
SELECT
    region,
    SUM(CASE WHEN status = 'CLOSED' THEN revenue ELSE 0 END) AS closed_revenue,
    SUM(CASE WHEN status = 'OPEN'   THEN revenue ELSE 0 END) AS open_revenue
FROM deals
GROUP BY region
ORDER BY closed_revenue DESC;""",
    },
    {
        "label": "Subquery with EXISTS",
        "sql": """\
SELECT customer_id, customer_name
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o
    WHERE o.customer_id = c.customer_id
    AND o.order_date >= '2024-01-01'
);""",
    },
    {
        "label": "NTILE window function",
        "sql": """\
SELECT
    employee_id,
    salary,
    NTILE(4) OVER (ORDER BY salary DESC) AS salary_quartile
FROM employees;""",
    },
    {
        "label": "Multi-table JOIN with DATE_TRUNC",
        "sql": """\
SELECT
    DATE_TRUNC('month', o.order_date) AS month,
    p.product_category,
    SUM(o.quantity * p.unit_price)    AS monthly_revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY 1, 2
ORDER BY 1, monthly_revenue DESC;""",
    },
    # --------------------------------------------------------
    # ADD YOUR OWN QUERIES BELOW THIS LINE
    # {
    #     "label": "My custom query",
    #     "sql": """\
    # SELECT ...
    # FROM ...
    # WHERE ...;""",
    # },
    # --------------------------------------------------------
]

# -----------------------------------------------------------
# Run all queries and print results
# -----------------------------------------------------------
print("=" * 65)
print("  Netezza SQL → PySpark Translation")
print(f"  {len(MY_QUERIES)} queries queued")
print("=" * 65)

for idx, entry in enumerate(MY_QUERIES, start=1):
    print(f"\n{'─' * 65}")
    print(f"  [{idx}/{len(MY_QUERIES)}]  {entry['label']}")
    print(f"{'─' * 65}")
    print("INPUT (Netezza SQL):")
    print(entry["sql"])
    print("\nOUTPUT (PySpark):")

    pyspark_code = translate(entry["sql"])
    print(pyspark_code)

print(f"\n{'=' * 65}")
print("  All translations complete.")
print("=" * 65)